# 🔍 EDA Masterclass

Complete Exploratory Data Analysis using the **Titanic dataset**. Uses `dskit.eda` for automation.

**Outline:** Load data → Auto-EDA → Missing values → Distributions → Correlations → Categorical analysis → Takeaways

In [1]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from dskit.eda import auto_eda, missing_report, outlier_summary, correlation_analysis
from dskit.viz import plot_missing_values, plot_correlation_matrix, plot_distributions

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 120
print('Setup complete ✅')

Setup complete ✅


## 1. Load & Inspect Data

In [2]:
df = sns.load_dataset('titanic')
print(f'Shape: {df.shape}')
df.head()

Shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.25,S
1,1,1,female,38.0,1,0,71.28,C
2,1,3,female,26.0,0,0,7.93,S
3,1,1,female,35.0,1,0,53.10,S
4,0,3,male,35.0,0,0,8.05,S


In [3]:
df.info()
print()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
10   adult_male   891 non-null    bool    
11   deck         203 non-null    object  
12   embark_town  889 non-null    object  
13   alive        891 non-null    object  
14   alone        891 non-null    bool    
dtypes: bool(2), category(1), float64(2), int64(4), object(5)
memory usage: 76.6+ KB


## 2. Auto-EDA in One Shot

In [4]:
report = auto_eda(df, target='survived')
print('Report keys:', list(report.keys()))
print()
print('Target analysis:', report['target_analysis'])

Report keys: ['shape', 'dtypes', 'missing', 'numeric_stats', 'categoricals', 'correlations', 'outliers', 'target_analysis']

Target analysis: {'type': 'classification', 'class_distribution': {0: 0.6162, 1: 0.3838}, 'n_classes': 2}


## 3. Missing Value Analysis

In [5]:
missing = missing_report(df)
print(f'Columns with missing values: {len(missing)}')
missing

Columns with missing values: 3


,missing_count,missing_pct
deck,688,77.22
age,177,19.87
embarked,2,0.22


In [6]:
fig = plot_missing_values(df)
plt.show()

<Figure size 900x400 with 1 Axes>

## 4. Distributions & Outliers

In [7]:
fig = plot_distributions(df, cols=['age', 'fare', 'sibsp', 'parch'])
plt.show()

<Figure size 1200x400 with 4 Axes>

In [8]:
for col in ['age', 'fare']:
    print(f'--- {col} ---')
    print(outlier_summary(df[col]))
    print()

--- age ---
{'iqr_outlier_count': 1, 'iqr_outlier_pct': 0.14, 'z_outlier_count': 1, 'z_outlier_pct': 0.14}

--- fare ---
{'iqr_outlier_count': 116, 'iqr_outlier_pct': 13.02, 'z_outlier_count': 20, 'z_outlier_pct': 2.25}


## 5. Correlation Heatmap

In [9]:
fig = plot_correlation_matrix(df)
plt.show()

corr = correlation_analysis(df)
print('Top correlations with survived:')
print(corr['survived'].drop('survived').abs().sort_values(ascending=False))

<Figure size 900x700 with 1 Axes>

Top correlations with survived:
pclass    0.338481
fare      0.257307
parch     0.081629
age       0.077221
sibsp     0.035322
dtype: float64


## 6. Categorical Analysis

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['sex', 'pclass', 'embarked']):
    rates = df.groupby(col)['survived'].mean().sort_values(ascending=False)
    rates.plot(kind='bar', ax=ax, color=sns.color_palette('husl', len(rates)), edgecolor='black')
    ax.set_title(f'Survival Rate by {col.title()}', fontweight='bold')
    ax.set_ylabel('Survival Rate')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=0)
plt.suptitle('Categorical Features vs Survival', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

<Figure size 1500x500 with 3 Axes>

## Key Takeaways

| Finding | Implication |
|---------|-------------|
| `age` ~20% missing | Impute before modelling |
| `fare` heavily right-skewed | Log transform recommended |
| `sex` strongest predictor | Female survival ~74%, male ~19% |
| `pclass` correlated with `fare` | Watch for multicollinearity |
| `deck` mostly missing | Drop this column |

➡️ **Next:** [02_feature_engineering.ipynb](02_feature_engineering.ipynb)